In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sum

In [2]:
# 1. Setting up Spark 
spark = SparkSession.builder \
    .appName("Walmart_Sales_Spark_Processing") \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .getOrCreate()

# Unified prefix path
HDFS_BASE = "hdfs://namenode:9000/user/walmart_sales/cleaned"
print("Reading Parquet files from HDFS...")

Reading Parquet files from HDFS...


In [3]:
# 2. Read the previously cleaned Parquet data
df_train = spark.read.parquet(f"{HDFS_BASE}/train")
df_features = spark.read.parquet(f"{HDFS_BASE}/features")
df_stores = spark.read.parquet(f"{HDFS_BASE}/stores")

In [4]:
# ==========================================
# Task 1: Conducting Joins (Connecting Multiple Tables to Integrate Large and Wide Tables)
# ==========================================
print("Conducting Joins...")

df_merged = df_train.join(df_features.drop("IsHoliday"), on=["Store", "Date"], how="inner")
df_merged = df_merged.join(df_stores, on=["Store"], how="inner")

Conducting Joins...


In [5]:
# ==========================================
# The impact of Markdown on sales data
# ==========================================
print("Conducting Markdown Impact SQL Queries...")

df_merged.createOrReplaceTempView("merged_sales")

sql_query_markdown = """
    SELECT 
        hasMarkDown1,
        COUNT(*) as record_count,
        SUM(Weekly_Sales) as Total_Sales,
        AVG(Weekly_Sales) as Avg_Sales
    FROM merged_sales
    GROUP BY hasMarkDown1
    ORDER BY Total_Sales DESC
"""
sql_markdown_results = spark.sql(sql_query_markdown)
sql_markdown_results.show()



Conducting Markdown Impact SQL Queries...
+------------+------------+--------------------+------------------+
|hasMarkDown1|record_count|         Total_Sales|         Avg_Sales|
+------------+------------+--------------------+------------------+
|           0|      267184|4.2388456704100757E9| 15864.89337089824|
|           1|      154386|2.4983733166999865E9|16182.641668933624|
+------------+------------+--------------------+------------------+



In [6]:
# ==========================================
# Task 2: Conducting Aggregations (Aggregation Calculation Based on DataFrame API)
# ==========================================
print("Conducting Aggregations...")
# Calculate the average weekly sales for each store and department
dept_avg_sales = df_merged.groupBy("Store", "Dept") \
    .agg(avg("Weekly_Sales").alias("Avg_Weekly_Sales"))
dept_avg_sales.show(5)


Conducting Aggregations...
+-----+----+------------------+
|Store|Dept|  Avg_Weekly_Sales|
+-----+----+------------------+
|    2|  80|26041.273566433563|
|    7|  55| 6616.540489510489|
|    8|  52|1945.2155944055958|
|   10|  85| 2933.471538461538|
|    3|  22|3101.7698601398597|
+-----+----+------------------+
only showing top 5 rows



In [7]:
# ==========================================
# Task 3: Conducting SQL Queries (Executing Complex Queries with Spark SQL)
# ==========================================
print("Conducting SQL Queries...")

df_merged.createOrReplaceTempView("merged_sales")
# Explore total sales on holidays and non-holidays, as well as by Store Type
sql_query = """
    SELECT 
        Type,
        IsHoliday,
        COUNT(*) as record_count,
        SUM(Weekly_Sales) as Total_Sales,
        AVG(Weekly_Sales) as Avg_Sales
    FROM merged_sales
    GROUP BY Type, IsHoliday
    ORDER BY Total_Sales DESC
"""
sql_results = spark.sql(sql_query)
sql_results.show()

Conducting SQL Queries...
+----+---------+------------+--------------------+------------------+
|Type|IsHoliday|record_count|         Total_Sales|         Avg_Sales|
+----+---------+------------+--------------------+------------------+
|   A|    false|      200293| 4.007611914590024E9|20008.746758948262|
|   B|    false|      151983|1.8470596961400008E9|12153.067751919629|
|   C|    false|       39633| 3.772478248199971E8| 9518.528115963896|
|   A|     true|       15185| 3.234028081599998E8| 21297.51782416857|
|   B|     true|       11512|1.5364104067999986E8|13346.164061848494|
|   C|     true|        2964| 2.825570272000001E7| 9532.963130904187|
+----+---------+------------+--------------------+------------------+



In [8]:
# ==========================================
# Task 4: Write results back to HDFS (save for visualization and MongoDB read)
# ==========================================
OUT_DIR = "hdfs://namenode:9000/user/walmart_sales/enriched"
print(f"Writing enriched data to {OUT_DIR}...")

Writing enriched data to hdfs://namenode:9000/user/walmart_sales/enriched...


In [9]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, avg, col

# ==========================================
# Task 5: Feature Engineering (Generating Temporal Lag Indicators and Rolling Averages)
# ==========================================
print("Conducting Feature Engineering...")

# Define the time series window: group by store and department, and strictly sort by date
windowSpec = Window.partitionBy("Store", "Dept").orderBy("Date")

# 1. Generate Lagged economic indicators
df_engineered = df_merged.withColumn("Prev_CPI", lag("CPI", 1).over(windowSpec)) \
                         .withColumn("Prev_Unemployment", lag("Unemployment", 1).over(windowSpec))

# 2. Generate rolling averages for the last 4 weeks
# rowsBetween(-4, -1) means 4 weeks forward, excluding this week
window_rolling_4wk = Window.partitionBy("Store", "Dept").orderBy("Date").rowsBetween(-4, -1)
df_engineered = df_engineered.withColumn("Avg_Sales_Prev_4Wk", avg("Weekly_Sales").over(window_rolling_4wk))

# 3. Fill the newly generated null values (e.g., no rolling average records for the first few weeks) to 0 to ensure that subsequent ML training does not report errors
df_engineered = df_engineered.fillna(0)

# Showcase the strong characteristics of the generation:
df_engineered.select("Store", "Dept", "Date", "Weekly_Sales", "Prev_CPI", "Avg_Sales_Prev_4Wk").show(5)


Conducting Feature Engineering...
+-----+----+----------+------------+-----------+------------------+
|Store|Dept|      Date|Weekly_Sales|   Prev_CPI|Avg_Sales_Prev_4Wk|
+-----+----+----------+------------+-----------+------------------+
|    1|   2|2010-02-05|    50605.27|       NULL|               0.0|
|    1|   2|2010-02-12|    44682.74|211.0963582|          50605.27|
|    1|   2|2010-02-19|    47928.89|211.2421698|         47644.005|
|    1|   2|2010-02-26|    44292.87|211.2891429| 47738.96666666667|
|    1|   2|2010-03-05|    48397.98|211.3196429|        46877.4425|
+-----+----+----------+------------+-----------+------------------+
only showing top 5 rows



In [11]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# ==========================================
# Task 6: Machine Learning Predictive Modeling (Spark MLlib Pipeline)
# ==========================================
print("Initializing Machine Learning Pipeline...")

# A. Convert the character categories Type (A, B, C) into a numerical index that ML can absorb
indexer = StringIndexer(inputCol="Type", outputCol="Type_Index")
df_ml = indexer.fit(df_engineered).transform(df_engineered)

# B. Force the Boolean type IsHoliday to be converted to a feature integer (1 or 0) to fill in the potential missing values
df_ml = df_ml.withColumn("IsHoliday_Int", col("IsHoliday").cast("integer")) \
             .withColumn("CPI", col("CPI").cast("double")) \
             .withColumn("Unemployment", col("Unemployment").cast("double")) \
             .withColumn("Prev_CPI", col("Prev_CPI").cast("double")) \
             .withColumn("Prev_Unemployment", col("Prev_Unemployment").cast("double")) \
             .fillna(0.0)

# C. VectorAssembler Core Assembly: Pack all features into a vector column as training set X
feature_cols = [
    "Store", "Dept", "Size", "Type_Index", "IsHoliday_Int",
    "Temperature", "Fuel_Price", "CPI", "Unemployment",
    "Prev_CPI", "Prev_Unemployment", "Avg_Sales_Prev_4Wk",
    "hasMarkDown1", "hasMarkDown2", "hasMarkDown3", "hasMarkDown4", "hasMarkDown5"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
final_data = assembler.transform(df_ml).select("features", "Weekly_Sales")

# D. Establish a Train/Test test set split (here we split our own wide table 80:20 to calculate RMSE)
train_data, test_data = final_data.randomSplit([0.8, 0.2], seed=42)

# E. Training Prediction Model: Random Forest Regression (numTrees and maxDepth can be improved on demand)
rf = RandomForestRegressor(featuresCol="features", labelCol="Weekly_Sales", numTrees=20, maxDepth=6, seed=42)
print("Training Random Forest Model (this will take 1-2 minutes)...")
rf_model = rf.fit(train_data)

# F. Test set prediction and evaluation validation using RMSE indicators
predictions = rf_model.transform(test_data)
evaluator = RegressionEvaluator(labelCol="Weekly_Sales", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"✅ Root Mean Squared Error (RMSE) on validation data: {rmse:.2f}")

# G. Feature Importance Reason Analysis (Analyze which features have the greatest impact on sales)
importances = rf_model.featureImportances
print("\n📊 Feature Importances (Top factors contributing to sales):")

# The feature influence coefficient is beautifully displayed through dictionary sorting
feature_score_map = {feature_cols[i]: float(importances[i]) for i in range(len(feature_cols))}
for feature, score in sorted(feature_score_map.items(), key=lambda x: x[1], reverse=True):
    print(f" - {feature:20}: {score:.5f}")


Initializing Machine Learning Pipeline...
Training Random Forest Model (this will take 1-2 minutes)...
✅ Root Mean Squared Error (RMSE) on validation data: 7877.48

📊 Feature Importances (Top factors contributing to sales):
 - Avg_Sales_Prev_4Wk  : 0.84592
 - Dept                : 0.08531
 - Size                : 0.03649
 - Type_Index          : 0.01619
 - Store               : 0.00823
 - CPI                 : 0.00275
 - Prev_Unemployment   : 0.00191
 - Prev_CPI            : 0.00168
 - Unemployment        : 0.00107
 - Temperature         : 0.00023
 - Fuel_Price          : 0.00011
 - IsHoliday_Int       : 0.00006
 - hasMarkDown2        : 0.00002
 - hasMarkDown1        : 0.00001
 - hasMarkDown5        : 0.00001
 - hasMarkDown4        : 0.00001
 - hasMarkDown3        : 0.00000


In [12]:
df_merged.write.mode("overwrite").parquet(f"{OUT_DIR}/merged_data")
sql_results.write.mode("overwrite").parquet(f"{OUT_DIR}/holiday_sales_analysis")
print("Spark Processing Complete!")
spark.stop()

Spark Processing Complete!
